In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def process_vod_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    VOD 데이터프레임을 일괄 정리하는 함수
    - FREE/무료 제거 및 플래그 생성
    - 자막/더빙 플래그 생성 및 괄호 제거
    - 예고 플래그 생성 및 괄호 제거
    - 숫자+회/화/부 텍스트 제거
    - 날짜(strt_dt) 분해 및 요일(한글) 생성
    - asset_id 추출
    - disp_rtm을 초(sec)로 변환
    - month 기반 season 생성
    - 불필요 컬럼 삭제
    """
    df = df.copy()
    
    # 1) FREE/무료 포함 여부 및 asset_nm 정리
    has_free_or_무료 = df['asset_nm'].str.contains(r'\((?:FREE|무료)\)', regex=True, na=False)
    df['asset_nm'] = df['asset_nm'].str.replace(r'\((?:FREE|무료)\)', '', regex=True)
    df['asset_nm'] = df['asset_nm'].str.replace(r'\s{2,}', ' ', regex=True).str.strip()
    df['asset_nm_free'] = has_free_or_무료.apply(lambda x: 0 if x else 1)
    mask_free = df['category'].str.contains(r'FREE|무료', flags=re.IGNORECASE, na=False)
    df.loc[mask_free, 'asset_nm_free'] = 0

    # 2) 자막/더빙 플래그 및 괄호 제거
    conds_trans = [
        df['asset_nm'].str.contains(r'\(.*자막.*\)', na=False),
        df['asset_nm'].str.contains(r'\(.*더빙.*\)', na=False)
    ]
    choices_trans = [1, 2]
    df['asset_nm_video_translation'] = np.select(conds_trans, choices_trans, default=0)
    df['asset_nm'] = df['asset_nm'].str.replace(r'\(.*(자막|더빙).*\)', '', regex=True).str.strip()

    # 3) 예고 플래그 및 괄호 제거
    df['asset_nm_preview'] = df['asset_nm'].str.contains('예고', na=False).astype(int)
    df['asset_nm'] = df['asset_nm'].str.replace(r'\([^)]*예고[^)]*\)', '', regex=True).str.strip()

    # 4) 숫자+회/화/부 및 모든 괄호 제거
    pattern1 = r'\([^)]*\)|\s*\d+회'
    pattern2 = r'\([^)]*\)|\s*\d+화'
    pattern3 = r'\([^)]*\)|\s*\d+부'
    df['asset_nm'] = (
        df['asset_nm']
        .str.replace(pattern1, '', regex=True)
        .str.replace(pattern2, '', regex=True)
        .str.replace(pattern3, '', regex=True)
        .str.replace(r'\s{2,}', ' ', regex=True)
        .str.strip()
    )

    # 5) strt_dt 컬럼 파싱 및 분해
    col = 'strt_dt'
    df[col] = df[col].fillna('00000000000000')
    df[col] = df[col].astype(str).str.replace('.0', '', regex=False).str.strip()
    df[f'{col}_dt'] = pd.to_datetime(df[col], format='%Y%m%d%H%M%S', errors='coerce')
    df['year'] = df[col].str[0:4].astype(int)
    df['month'] = df[col].str[4:6].astype(int)
    df['day'] = df[col].str[6:8].astype(int)
    df['hour'] = df[col].str[8:10].astype(int)
    df['minute'] = df[col].str[10:12].astype(int)
    df['second'] = df[col].str[12:14].astype(int)
    df['weekday_kr'] = df[f'{col}_dt'].dt.dayofweek.map({
        0: '월요일', 1: '화요일', 2: '수요일', 3: '목요일',
        4: '금요일', 5: '토요일', 6: '일요일'
    })

    # 6) asset_id 추출
    extracted = df['asset'].str.extract(r'\|(.+)$')[0]
    df['asset_id'] = extracted.fillna(df['asset'])

    # 7) disp_rtm을 초(sec)로 변환
    td = pd.to_timedelta(df['disp_rtm'].fillna("00:00") + ":00", errors='coerce')
    df['disp_rtm'] = td.dt.total_seconds().fillna(0).astype(int)

    # 8) season 생성 (기상학적 기준)
    conds_season = [
        df['month'].between(3, 5),
        df['month'].between(6, 8),
        df['month'].between(9, 11),
        df['month'].isin([12, 1, 2])
    ]
    choices_season = [0, 1, 2, 3] # 봄=0, 여름=1, 가을=2, 겨울=3 입니다
    df['season'] = np.select(conds_season, choices_season, default=np.nan)

    # 9) 불필요 컬럼 삭제
    df = df.drop(columns=['year', 'minute', 'second'], errors='ignore') 

    return df

def process_dt(df: pd.DataFrame) -> pd.DataFrame:
    # 5) strt_dt 컬럼 파싱 및 분해
    col = 'strt_dt'
    df[col] = df[col].fillna('00000000000000')
    df[col] = df[col].astype(str).str.replace('.0', '', regex=False).str.strip()
    df[f'{col}_dt'] = pd.to_datetime(df[col], format='%Y%m%d%H%M%S', errors='coerce')
    df['year'] = df[col].str[0:4].astype(int)
    df['month'] = df[col].str[4:6].astype(int)
    df['day'] = df[col].str[6:8].astype(int)
    df['hour'] = df[col].str[8:10].astype(int)
    df['minute'] = df[col].str[10:12].astype(int)
    df['second'] = df[col].str[12:14].astype(int)
    df['weekday_kr'] = df[f'{col}_dt'].dt.dayofweek.map({
        0: '월요일', 1: '화요일', 2: '수요일', 3: '목요일',
        4: '금요일', 5: '토요일', 6: '일요일'
    })

    # 7) disp_rtm을 초(sec)로 변환
    td = pd.to_timedelta(
        df['disp_rtm'].astype(str).fillna(np.nan) + ":00",
        errors='coerce'
    )

    # 2) total_seconds() 로 초 단위 float 얻기 (NaT → NaN)
    df['disp_rtm_s'] = td.dt.total_seconds()

    # 8) season 생성 (기상학적 기준)
    conds_season = [
        df['month'].between(3, 5),
        df['month'].between(6, 8),
        df['month'].between(9, 11),
        df['month'].isin([12, 1, 2])
    ]
    choices_season = [0, 1, 2, 3] # 봄=0, 여름=1, 가을=2, 겨울=3 입니다
    df['season'] = np.select(conds_season, choices_season, default=np.nan)

    # 9) 불필요 컬럼 삭제
    df = df.drop(columns=['year', 'minute', 'second'], errors='ignore') 

    return df

def split_category_levels(df: pd.DataFrame, source_col: str = 'category', max_levels: int = 4) -> pd.DataFrame:
    """
    'category' 컬럼의 값을 '/'로 분리하여 category_l1 ~ category_l4 컬럼에 할당합니다.
    
    Args:
        df (pd.DataFrame): 원본 데이터프레임
        source_col (str): 분할할 원본 컬럼명 (기본값 'category')
        max_levels (int): 생성할 레벨 개수 (기본값 4)
    
    Returns:
        pd.DataFrame: category_l1 ~ category_l4 컬럼이 추가된 데이터프레임
    """
    # 문자열을 '/' 기준으로 분할, expand=True로 DataFrame으로 반환
    parts = df[source_col].str.split('/', expand=True)
    
    # 각 레벨 컬럼 생성
    for i in range(max_levels):
        # 존재하지 않는 레벨은 NaN -> 빈 문자열로 대체
        df[f'category_l{i+1}'] = parts[i].fillna(np.nan)
    
    return df

def add_free_column(df: pd.DataFrame) -> pd.DataFrame:
    """
    category_l1 이 '프리미엄 무료관' 이거나
    category_l2 이 '무료영화', 'FOD시연입수' 인 경우
    또는 제목에 '깜짝무료'가 포함되어 있거나
    제목이 '해적-바다로 간 산적(8월 무료)'인 경우 free=0,
    나머지는 free=1 이 되도록 `free` 컬럼을 추가합니다.
    """
    df['free'] = np.where(
        (df['category_l1'] == '프리미엄 무료관') |
        (df['category_l2'] == '무료영화') |
        (df['category_l2'] == 'FOD시연입수') |
        (df['asset_nm'].str.contains('깜짝무료', na = False)) |
        (df['asset_nm'] == '해적-바다로 간 산적(8월 무료)'),        
        0,
        1
    )
    return df
def add_kids_column(df: pd.DataFrame) -> pd.DataFrame:

    df['kids'] = np.where(
        (df['category_l1'] == '키즈어린이') |
        (df['category_l2'] == '무료 키즈관') |
        (df['category_l2'] == '학원,순정어린이') |
        (df['category_l2'] == '액션모험어린이') |
        (df['category_l2'] == '코믹어린이') |
        (df['category_l2'] == '추리,판타지어린이') |
        (df['ct_cl'] == '키즈'),        
        0,
        1
    )
    return df




In [5]:

vodmart = pd.read_csv('./Downloads/LG헬로비전 데이터/241224 3기 추가 데이터 1/vod_mart_data.csv', on_bad_lines='skip' )
split_category_levels(vodmart)
add_kids_column(vodmart)

df = pd.read_csv('./Downloads/LG헬로비전 데이터/241224 3기 추가 데이터 1/202301_VOD.csv')
split_category_levels(df)
df.rename(columns = {'CT_CL' : 'ct_cl',
                    'asset' : 'full_asset_id'}, inplace = True)
add_kids_column(df)



C:\Users\user\AppData\Local\Temp\ipykernel_15968\281374719.py:1: DtypeWarning: Columns (6,40,41,43,78,79,80) have mixed types. Specify dtype option on import or set low_memory=False.
  vodmart = pd.read_csv('./Downloads/LG헬로비전 데이터/241224 3기 추가 데이터 1/vod_mart_data.csv', on_bad_lines='skip' )


,sha2_hash,full_asset_id,asset_nm,ct_cl,genre_of_ct_cl,use_tms,disp_rtm,strt_dt,category,category_l1,category_l2,category_l3,category_l4,kids
0,992c0dd6bafc5df33e86ece4885d574d25288d06530c65...,cjc|M4767613LSGK41566601,날 녹여주오 04회,TV드라마,외화 시리즈,3660.0,01:01,2.023012e+13,CJ ENM/CJENM구작/날 녹여주오,CJ ENM,CJENM구작,날 녹여주오,NaN,1
1,ea62338ac5b6b11cf02ef8bf1889d1a063cec2c2493937...,cjc|M5140475LSGL08601501,압꾸정,영화,코미디,2.0,01:51,2.023012e+13,영화/(HD)극장동시상영관,영화,(HD)극장동시상영관,NaN,NaN,1
2,36aa302f3705794f5c2e5a971f38c8ef3c5a789e915c94...,cjc|M5079740LSVK14589601,어웨이크(2021),영화,공포/스릴러,7.0,01:14,2.023012e+13,영화월정액관/무비n시리즈/범죄공포,영화월정액관,무비n시리즈,범죄공포,NaN,1
3,d1ff76342bbc4f23f82c318b1f7a0ffc78f0b5ad2d1314...,cjc|M4574740LFOL08934201,동글동글 동물친구 시즌2 01회,키즈,기타,60.0,00:01,2.023012e+13,키즈어린이/영어-놀이학습/동글동글 동물친구 시즌2,키즈어린이,영어-놀이학습,동글동글 동물친구 시즌2,NaN,0
4,8cf3e070a173520f59e24b246ef83a1140c26d260c4ddd...,cjc|M5055522LFOI39238101,1박2일 시즌4 127회(22/05/29),TV 연예/오락,기타,157.0,01:19,2.023012e+13,KBS/(HD)KBS 연예오락/1박2일 시즌4,KBS,(HD)KBS 연예오락,1박2일 시즌4,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6325393,95e2cfa19653d4d03bc635128a6de294fa057454c93676...,cjc|M4740025LSGK16146201,의사 요한 02회,TV드라마,기타,3600.0,01:00,2.023012e+13,SBS/SBS구작/의사 요한,SBS,SBS구작,의사 요한,NaN,1
6325394,a4d8febf58720ecc6d8f758038eb73ab6a68e79419fcfd...,cjc|M5042784LFOJ31764601,(HD)런닝맨 634회(22/12/25),TV 연예/오락,기타,4291.0,01:26,2.023012e+13,SBS/(HD)SBS 연예오락/(HD)런닝맨,SBS,(HD)SBS 연예오락,(HD)런닝맨,NaN,1
6325395,034d6981d3b7b60ca66bd72080ef8b271912763522bd64...,cjc|M4996864LFOL10619201,(FREE)완전한 사육: 욕망의 시작(무료),영화,드라마,317.0,01:21,2.023012e+13,프리미엄 무료관/무료영화/무료 영화관,프리미엄 무료관,무료영화,무료 영화관,NaN,1
6325396,f638c91cbd3488fda7a40dd7624ea3995bab6acc51f4fb...,cjc|M5143658LFOL23041801,만지지 마세요!,영화,애니메이션,40.0,00:09,2.023012e+13,프리미엄 무료관/무료영화/무료 영화관,프리미엄 무료관,무료영화,무료 영화관,NaN,1


In [27]:

mart_kids = vodmart[vodmart["kids"] == 0].copy()

max_sq = vodmart['series_sq'].dropna().astype(int).max()
offset = max_sq + 1

fill_values = pd.Series(
    data=mart_kids.index + offset,
    index=mart_kids.index
)

mart_kids["series_sq"] = mart_kids["series_sq"].fillna(fill_values).astype(int)
mart_kids["epsd_no"] = mart_kids["epsd_no"].fillna(1)

In [9]:
vod_kids = (
    df.merge(
        mart_kids[['full_asset_id', 'series_sq']],
        on = 'full_asset_id', how = 'inner'
    )
)

In [11]:
# 2) (sha2_hash, series_sq)별 로그 개수 세기
watch_counts = (
    vod_kids
    .groupby(['sha2_hash', 'series_sq'])
    .size()
    .reset_index(name='view_count')
)

In [71]:
watch_counts.head()

,sha2_hash,series_sq,view_count
0,000235ee2567ae4c3cca998fbc998dbdc14f43e074dc4b...,72128,10
1,0006a180b88a5c6cb289a7687e0bacdcc06346e7845eb9...,2244215,1
2,000783d9b8c3941733d01dfd58425b7e8541f4c25ef15d...,985582,1
3,000937c969b88977787e2c24dd86eeae39d10805e60942...,1272085,1
4,000b8f1b6ce45495467f66138676417b225b8f9099633c...,2242973,1


In [75]:
watch_counts[vod_counts['sha2_hash'].str.contains('000b8f1b6ce45495467f66138676417b225b8f9099633c')]

,sha2_hash,series_sq,view_count
4,000b8f1b6ce45495467f66138676417b225b8f9099633c...,2242973,1
5,000b8f1b6ce45495467f66138676417b225b8f9099633c...,2242979,1
6,000b8f1b6ce45495467f66138676417b225b8f9099633c...,2243015,1
7,000b8f1b6ce45495467f66138676417b225b8f9099633c...,2243038,1
8,000b8f1b6ce45495467f66138676417b225b8f9099633c...,2243065,1
9,000b8f1b6ce45495467f66138676417b225b8f9099633c...,2243095,1
10,000b8f1b6ce45495467f66138676417b225b8f9099633c...,2243104,1
11,000b8f1b6ce45495467f66138676417b225b8f9099633c...,2243363,1
12,000b8f1b6ce45495467f66138676417b225b8f9099633c...,2243364,1
13,000b8f1b6ce45495467f66138676417b225b8f9099633c...,2243370,1


In [58]:
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import coo_matrix
import implicit

user_enc = LabelEncoder().fit(watch_counts['sha2_hash'])
item_enc = LabelEncoder().fit(watch_counts['series_sq'])

watch_counts['user_idx'] = user_enc.transform(watch_counts['sha2_hash'])
watch_counts['item_idx'] = item_enc.transform(watch_counts['series_sq'])

# --- interactions 생성 ---
n_users = len(user_enc.classes_)
n_items = len(item_enc.classes_)

interactions = coo_matrix(
    (watch_counts['view_count'],
     (watch_counts['user_idx'], watch_counts['item_idx'])),
    shape=(n_users, n_items)
).tocsr()

# --- ALS 학습 ---
model = implicit.als.AlternatingLeastSquares(factors=50, regularization=0.01, iterations=20)
model.fit(interactions)

def print_rec_result(result):
    rec_vods = mart_kids[
        mart_kids['series_sq'].isin(result)
    ][['series_sq', 'asset_nm', 'smry', 'epsd_no']].copy()
    
    rec_first_epsd = (
        rec_vods
        .sort_values(['series_sq', 'epsd_no'])
        .drop_duplicates(subset='series_sq', keep = 'first')
        .reset_index(drop = True)
    )
    
    return print(rec_first_epsd)

# --- 추천 함수 ---
def recommend_for_hash(user_hash, N=5):
    if user_hash not in user_enc.classes_:
        return []

    uid = user_enc.transform([user_hash])[0]
    user_items = interactions[uid, :]   # 이제 안전
    rec_ids, rec_scores = model.recommend(
        userid=uid,
        user_items=user_items,
        N=N,
        filter_already_liked_items=True
    )
    result = item_enc.inverse_transform(rec_ids).tolist()
    return print_rec_result(result)


  0%|          | 0/20 [00:00<?, ?it/s]

In [46]:
!pip install implicit

  Installing build dependencies: started
  Installing build dependencies: still running...
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for implicit: filename=implicit-0.7.2-cp312-cp312-win_amd64.whl size=753357 sha256=3846fac7858d73633d9953e0051d81b47bbe934ed6bd8b8160321c0553566c84
  Stored in directory: c:\users\user\appdata\local\pip\cache\wheels\b2\00\4f\9ff8af07a0a53ac6007ea5d739da19cfe147a2df542b6899f8
Successfully built implicit
  Using cached implicit-0.7.2.tar.gz (70 kB)
  Installing build dependencies: started
  Installing build dependencies: still running...
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with sta

ERROR: Could not install packages due to an OSError: [WinError 32] 다른 프로세스가 파일을 사용 중이기 때문에 프로세스가 액세스 할 수 없습니다: 'C:\\Users\\user\\anaconda3\\Lib\\site-packages\\implicit\\_nearest_neighbours.cp312-win_amd64.pyd'
Consider using the `--user` option or check the permissions.

ERROR: Could not install packages due to an OSError: [WinError 32] 다른 프로세스가 파일을 사용 중이기 때문에 프로세스가 액세스 할 수 없습니다: 'C:\\Users\\user\\anaconda3\\Lib\\site-packages\\implicit\\_nearest_neighbours.cp312-win_amd64.pyd'
Consider using the `--user` option or check the permissions.



In [52]:
vodmart['series_sq'].max()

2065893.0

In [21]:
result = recommend_for_hash(user_hash, N = 5).tolist()

In [37]:
rec_vods = mart_kids[
    mart_kids['series_sq'].isin(result)
][['series_sq', 'asset_nm', 'smry', 'epsd_no']].copy()

rec_first_epsd = (
    rec_vods
    .sort_values(['series_sq', 'epsd_no'])
    .drop_duplicates(subset='series_sq', keep = 'first')
    .reset_index(drop = True)
)

print(rec_first_epsd)

   series_sq                                  asset_nm  \
0     985582  The Wheels on the Bus Go Round and Round   
1    1587588                   뽀로로 병원에 놀러가요(영어) 01회(P)   
2    2244212                           늑대와 일곱 마리 아기 염소   
3    2244213                                       라푼젤   
4    2244214                                      신데렐라   

                                                smry  epsd_no  
0  그림책 속 세상을 마음껏 여행하는 책갈피 요정 또보. 빵빵! 하고 들려오는 이 소리...      1.0  
1  Emergency Room Song. 뽀로로 병원에 응급환자가 왔어요, 어서 빨리 ...      1.0  
2  [키즈스콜레] 늑대가 엄마로 변장하여 아기 염소들을 잡아먹었어요. 일곱째 아기 염소...      1.0  
3  [키즈스콜레] 탑에 갇힌 라푼젤을 구하기 위해 왕자는 매일 밤 비단실을 가져다줍니다...      1.0  
4  [키즈스콜레]신데렐라는 새어머니와 언니들에게 구박을 받으며 살아요. 요정 덕분에 파...      1.0  


In [42]:
recommend_for_hash('e53c6eb15ce65583b11570e9573bdcbad8df21616d1d36d7b2b7e27e90543277')

   series_sq                   asset_nm  \
0     957390  (TV유치원)지니와 직업탐험 바쁘다바빠 01회   
1     974282                         거미   
2     974292                     감자도시 외   
3     974295                       출동이다   
4    1662484                   아기상어와 과일   

                                                smry  epsd_no  
0       아쿠아리스트 (1) 아쿠아리스트로 변신한 지니! 아쿠아리스트는 어떤 일을 할까?      1.0  
1  [안내] 핑크퐁! 3D율동동요 시리즈가 아쉽게도 23년 3월 31일 까지 무료 제공...      1.0  
2  1. 감자 도시. 채소의 마법이 끝없이 펼쳐지는 감자도시(테마파크)에 놀러간 페파 ...      1.0  
3  [안내] 핑크퐁! 자동차동요 시리즈가 아쉽게도 23년 3월 31일 까지 무료 제공 ...      1.0  
4  아기상어와 과일. 과일이 먹고 싶은 올리가 으앙 울고 있어요! 호기는 섬에 있는 과...      1.0  


In [45]:
recommend_for_hash('796f8754cbd26f4750037ec3287093674fb711deaa30f5a4b39c602f99afd35d')

   series_sq               asset_nm  \
0      25954            라바 인 뉴욕 01회   
1      69844             라바 시즌1 01회   
2      72128    뽀로로와 노래해요 NEW 1 01회   
3     697596      바다탐험대 옥토넛 시즌3 01회   
4     992806  바다 탐험대 옥토넛 탐험선 보고 01회   

                                                smry  epsd_no  
0  도넛. 외롭고 쓸쓸한 뉴욕의 거리로 나온 레드와 옐로. 배고픔에 지쳐 쓰러져가는 그...      1.0  
1  1. 아이스크림. 2. 모기. 3. 탭댄스. 4. 버섯. 5. 껌. 6. 빙판. 7...      1.0  
2  개구리. 뽀로로와 친구들이 무대에서 공연을 해요. 그런데 에디가 너무 신이나서 노래...      1.0  
3  물곰 The Water Bears. 용암동굴의 온도를 조사하러 나갔던 바나클과 페이...      1.0  
4  트윅이 소개하는 옥토 심해 연구실 1편! 연구실이 어떻게 만들어졌는지 알려주고, 연...      1.0  


In [51]:
recommend_for_hash('23f6a8503c9ba88fefdf96699f82facf2d9e4abf401c1cd7d0b388b1463586fb')

   series_sq                                  asset_nm  \
0     985582  The Wheels on the Bus Go Round and Round   
1    1587588                   뽀로로 병원에 놀러가요(영어) 01회(P)   
2    2244212                           늑대와 일곱 마리 아기 염소   
3    2244213                                       라푼젤   
4    2244214                                      신데렐라   

                                                smry  epsd_no  
0  그림책 속 세상을 마음껏 여행하는 책갈피 요정 또보. 빵빵! 하고 들려오는 이 소리...      1.0  
1  Emergency Room Song. 뽀로로 병원에 응급환자가 왔어요, 어서 빨리 ...      1.0  
2  [키즈스콜레] 늑대가 엄마로 변장하여 아기 염소들을 잡아먹었어요. 일곱째 아기 염소...      1.0  
3  [키즈스콜레] 탑에 갇힌 라푼젤을 구하기 위해 왕자는 매일 밤 비단실을 가져다줍니다...      1.0  
4  [키즈스콜레]신데렐라는 새어머니와 언니들에게 구박을 받으며 살아요. 요정 덕분에 파...      1.0  
